In [10]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

brand_df = pd.read_pickle("../data_csv/brand_analysis_result.pkl").reset_index(drop=True)
persona_sim = pd.read_csv("../data_csv/persona_product_similarity_full.csv")

brand_df["embedding_vector"] = brand_df["embedding_vector"].apply(np.array)

brand_mean_df = (
    brand_df
    .groupby("브랜드")["embedding_vector"]
    .apply(lambda x: np.mean(np.vstack(x.values), axis=0))
    .reset_index()
    .rename(columns={"브랜드": "brand", "embedding_vector": "vector"})
)

X = np.vstack(brand_mean_df["vector"].values)
k = min(4, len(brand_mean_df))

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

brand_mean_df["brand_tone_cluster"] = kmeans.fit_predict(X)

CLUSTER_TO_POSITION = {
    0: "고기능/클리니컬 톤",
    1: "수분 중심 톤",
    2: "프리미엄/헤리티지 톤",
    3: "트렌디/컬러 중심 톤"
}

brand_mean_df["brand_position"] = (
    brand_mean_df["brand_tone_cluster"]
    .map(CLUSTER_TO_POSITION)
)

brand_segment = brand_mean_df[
    ["brand", "brand_tone_cluster", "brand_position"]
]

brand_segment.to_csv(
    "../data_csv/final_brand_segment.csv",
    index=False
)

persona_brand = (
    persona_sim
    .merge(
        brand_mean_df[["brand", "brand_tone_cluster"]],
        on="brand",
        how="left"
    )
    .groupby(["persona_id", "brand", "brand_tone_cluster"])
    .agg(
        avg_similarity=("similarity", "mean"),
        product_cnt=("product_index", "count")
    )
    .reset_index()
)

persona_brand.to_csv(
    "../data_csv/persona_brand_tone_strategy.csv",
    index=False
)

print(persona_sim["persona_id"].value_counts())
print(persona_sim.groupby("persona_id")["brand"].nunique())

persona_id
persona_1    1581
persona_2    1581
persona_3    1581
Name: count, dtype: int64
persona_id
persona_1    25
persona_2    25
persona_3    25
Name: brand, dtype: int64
